# Tutorial: Compare Aligned Spaces

Audience:
- Python users comparing two views over the same observations.

Prerequisites:
- Stable record IDs and callable metrics.

Learning goals:
- Compare spaces by position and by IDs.
- Handle reordered observations.
- Read correlation result metadata.

## Outline

1. Import the public facade.
2. Build two aligned spaces.
3. Compare by position.
4. Compare by stable IDs.
5. Try a small exercise.

In [ ]:
from metric import Space
from metric import metrics


## Step 1 - Build source and quality spaces

Both spaces describe the same observations with different record values.

In [ ]:
ids = ["run-a", "run-b", "run-c", "run-d"]
process = Space([0, 1, 2, 3], metric=lambda lhs, rhs: abs(lhs - rhs), ids=ids)
quality = Space([0, 1, 4, 9], metric=lambda lhs, rhs: abs(lhs - rhs), ids=ids)

assert process.ids == quality.ids
len(process.pairwise())


## Step 2 - Compare by position

The default compares records in their current order.

In [ ]:
from metric.exceptions import StrategyUnavailableError

try:
    process.compare(quality)
    raise AssertionError("compare should require a native binding")
except StrategyUnavailableError as exc:
    message = str(exc)

assert "native C++ binding" in message
message


## Step 3 - Compare by IDs

ID alignment keeps the comparison stable even when the right-hand space is reordered.

In [ ]:
shuffled_quality = Space(
    [9, 0, 4, 1],
    metric=lambda lhs, rhs: abs(lhs - rhs),
    ids=["run-d", "run-a", "run-c", "run-b"],
)

for call in (
    lambda: process.compare(shuffled_quality, align="ids"),
    lambda: process.compare(shuffled_quality, align="position"),
):
    try:
        call()
        raise AssertionError("compare should require a native binding")
    except StrategyUnavailableError:
        pass
"compare requires native C++ binding"


## Exercise

Remove one ID from `shuffled_quality` and confirm that ID alignment fails.

Common pitfall: `align="ids"` requires both spaces to contain exactly the same IDs.

In [ ]:
mismatched = Space([0, 1, 4], metric=lambda lhs, rhs: abs(lhs - rhs), ids=["run-a", "run-b", "run-x"])

try:
    process.compare(mismatched, align="ids")
    raise AssertionError("compare should require a native binding")
except StrategyUnavailableError as exc:
    message = str(exc)

assert "native C++ binding" in message
message
